# Corrected WPT Robustness Project

Run the code cell below in Google Colab. It generates the full corrected project, saves all figures and CSV files, includes source code inside the ZIP, and auto-downloads the result ZIP at the end.

Corrections included:
- C2 and L2 detuning sweeps
- Frequency tracking under combined misalignment + detuning
- Load optimization by maximum efficiency, maximum output power, and constrained efficiency subject to output-power target
- Efficiency and output power plotted together
- Robustness thresholds of 50%, 60%, and 70% with justification
- Corrected coupling interpretation: `k_ref` is defined at `d_ref`
- Full source code included inside the final ZIP


In [ ]:
# Complete corrected Google Colab-friendly WPT project pipeline
# Run this file/cell from beginning to end. It writes the source code, executes the project,
# generates figures/CSV/report files, creates a ZIP, and auto-downloads the ZIP in Colab.

from pathlib import Path
import runpy
import sys

PROJECT_ROOT = Path("WPT_Robustness_Project_Corrected")
CODE_DIR = PROJECT_ROOT / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "report").mkdir(parents=True, exist_ok=True)

ADVANCED_WPT_MODEL_CODE = '\n"""\nAdvanced two-coil resonant wireless power transfer model.\n\nThis module intentionally keeps the physics at equivalent-circuit level.\nIt is suitable for a reproducible student simulation project, not for\nfull electromagnetic coil design or experimental validation.\n"""\n\nfrom dataclasses import dataclass\nimport numpy as np\n\n\n@dataclass\nclass WPTParams:\n    """Parameter container for a two-coil series-series resonant WPT system.\n\n    Important interpretation of the coupling parameters:\n    - k_ref is the coupling coefficient at the reference distance d_ref when\n      lateral shift = 0 and tilt = 0.\n    - The distance model is normalized so that coupling_distance(d_ref) = k_ref.\n      This fixes the common ambiguity where k0 is accidentally treated as both\n      the zero-distance coupling and the reference-distance coupling.\n    """\n\n    R1: float = 0.5          # transmitter coil resistance, ohm\n    R2: float = 0.5          # receiver coil resistance, ohm\n    L1: float = 100e-6       # transmitter inductance, H\n    L2: float = 100e-6       # receiver inductance, H\n    C1: float = 100e-9       # transmitter compensation capacitance, F\n    C2: float = 100e-9       # receiver compensation capacitance, F\n    Vs: float = 10.0         # RMS source voltage, V\n    k_ref: float = 0.35      # coupling coefficient at d_ref, x=0, theta=0\n    d_ref: float = 0.05      # reference distance, m\n    n_decay: float = 2.0     # distance-decay exponent\n    coil_radius: float = 0.08 # coil radius, m\n    k_max: float = 0.95      # physical safety clip for coupling coefficient\n\n\ndef scaled_params(params: WPTParams, C2_scale: float = 1.0, L2_scale: float = 1.0,\n                  C1_scale: float = 1.0, L1_scale: float = 1.0) -> WPTParams:\n    """Return a copy of params with selected L/C detuning factors applied."""\n    return WPTParams(\n        R1=params.R1,\n        R2=params.R2,\n        L1=params.L1 * L1_scale,\n        L2=params.L2 * L2_scale,\n        C1=params.C1 * C1_scale,\n        C2=params.C2 * C2_scale,\n        Vs=params.Vs,\n        k_ref=params.k_ref,\n        d_ref=params.d_ref,\n        n_decay=params.n_decay,\n        coil_radius=params.coil_radius,\n        k_max=params.k_max,\n    )\n\n\ndef resonance_frequency(L: float, C: float) -> float:\n    """LC resonance frequency in hertz."""\n    if L <= 0 or C <= 0:\n        raise ValueError("L and C must be positive.")\n    return 1.0 / (2.0 * np.pi * np.sqrt(L * C))\n\n\ndef mutual_inductance(k: float, params: WPTParams) -> float:\n    """Mutual inductance from coupling coefficient."""\n    return float(k) * np.sqrt(params.L1 * params.L2)\n\n\ndef coupling_distance(d: float, params: WPTParams) -> float:\n    """Distance-dependent coupling coefficient normalized at d_ref.\n\n    Model:\n        k(d) = k_ref * [1 + (d_ref/d_ref)^n] / [1 + (d/d_ref)^n]\n             = k_ref * 2 / [1 + (d/d_ref)^n]\n\n    Therefore k(d_ref) = k_ref. This is a trend model, not an FEM-derived coil\n    model.\n    """\n    if d < 0:\n        raise ValueError("Distance d must be non-negative.")\n    normalized = (1.0 + 1.0 ** params.n_decay) / (1.0 + (d / params.d_ref) ** params.n_decay)\n    return float(np.clip(params.k_ref * normalized, 0.0, params.k_max))\n\n\ndef coupling_lateral_factor(x: float, params: WPTParams) -> float:\n    """Empirical lateral misalignment reduction factor."""\n    if x < 0:\n        raise ValueError("Lateral shift x must be non-negative.")\n    return float(np.exp(- (x / params.coil_radius) ** 2))\n\n\ndef coupling_tilt_factor(theta_deg: float) -> float:\n    """Empirical angular tilt reduction factor."""\n    return float(max(np.cos(np.deg2rad(theta_deg)), 0.0))\n\n\ndef combined_coupling(d: float, x: float, theta_deg: float, params: WPTParams) -> float:\n    """Combined effective coupling under distance, lateral shift, and tilt."""\n    k_eff = coupling_distance(d, params) * coupling_lateral_factor(x, params) * coupling_tilt_factor(theta_deg)\n    return float(np.clip(k_eff, 0.0, params.k_max))\n\n\ndef solve_circuit(frequency: float, k: float, RL: float, params: WPTParams) -> dict:\n    """Solve the two-coil series-series WPT circuit.\n\n    The source voltage is treated as an RMS phasor. The input power is\n    Re(Vs * conj(I1)). Output power is |I2|^2 RL. The function returns both\n    efficiency and output power so optimization is not judged by efficiency alone.\n    """\n    if frequency <= 0:\n        raise ValueError("frequency must be positive.")\n    if RL <= 0:\n        raise ValueError("RL must be positive.")\n    if not (0.0 <= k < 1.0):\n        raise ValueError("k must satisfy 0 <= k < 1.")\n\n    omega = 2.0 * np.pi * frequency\n    M = mutual_inductance(k, params)\n\n    Z1 = params.R1 + 1j * (omega * params.L1 - 1.0 / (omega * params.C1))\n    Z2 = params.R2 + RL + 1j * (omega * params.L2 - 1.0 / (omega * params.C2))\n\n    A = np.array([[Z1, 1j * omega * M], [1j * omega * M, Z2]], dtype=complex)\n    b = np.array([params.Vs, 0.0], dtype=complex)\n    I1, I2 = np.linalg.solve(A, b)\n\n    Pin = float(np.real(params.Vs * np.conj(I1)))\n    Pout = float((abs(I2) ** 2) * RL)\n    eta = float(Pout / Pin) if Pin > 1e-15 else 0.0\n\n    # Small numerical artifacts can occur near singular regions; keep values honest.\n    if eta < 0 and eta > -1e-12:\n        eta = 0.0\n\n    return {\n        "I1": I1,\n        "I2": I2,\n        "Pin": Pin,\n        "Pout": Pout,\n        "eta": eta,\n        "eta_percent": 100.0 * eta,\n        "Z1": Z1,\n        "Z2": Z2,\n        "M": M,\n        "k": k,\n    }\n\n\ndef frequency_sweep(frequencies, k: float, RL: float, params: WPTParams):\n    """Sweep frequency and return arrays of efficiency, output power, and input power."""\n    eta_values = []\n    pout_values = []\n    pin_values = []\n    for f in frequencies:\n        result = solve_circuit(float(f), k, RL, params)\n        eta_values.append(result["eta"])\n        pout_values.append(result["Pout"])\n        pin_values.append(result["Pin"])\n    return np.asarray(eta_values), np.asarray(pout_values), np.asarray(pin_values)\n\n\ndef find_best_frequency(frequencies, k: float, RL: float, params: WPTParams) -> dict:\n    """Find frequency that gives maximum efficiency for fixed k and RL."""\n    eta_values, pout_values, pin_values = frequency_sweep(frequencies, k, RL, params)\n    idx = int(np.nanargmax(eta_values))\n    return {\n        "f_best": float(frequencies[idx]),\n        "eta_best": float(eta_values[idx]),\n        "eta_best_percent": float(100.0 * eta_values[idx]),\n        "pout_at_best": float(pout_values[idx]),\n        "pin_at_best": float(pin_values[idx]),\n        "index": idx,\n    }\n\n\ndef load_sweep(load_values, frequency: float, k: float, params: WPTParams):\n    """Sweep load resistance and return arrays of efficiency, output power, and input power."""\n    eta_values = []\n    pout_values = []\n    pin_values = []\n    for RL in load_values:\n        result = solve_circuit(frequency, k, float(RL), params)\n        eta_values.append(result["eta"])\n        pout_values.append(result["Pout"])\n        pin_values.append(result["Pin"])\n    return np.asarray(eta_values), np.asarray(pout_values), np.asarray(pin_values)\n\n\ndef find_best_load_max_eta(load_values, frequency: float, k: float, params: WPTParams) -> dict:\n    """Find load that maximizes efficiency."""\n    eta_values, pout_values, pin_values = load_sweep(load_values, frequency, k, params)\n    idx = int(np.nanargmax(eta_values))\n    return {\n        "RL_best": float(load_values[idx]),\n        "eta_best": float(eta_values[idx]),\n        "eta_best_percent": float(100.0 * eta_values[idx]),\n        "pout_at_best": float(pout_values[idx]),\n        "pin_at_best": float(pin_values[idx]),\n        "index": idx,\n    }\n\n\ndef find_best_load_max_pout(load_values, frequency: float, k: float, params: WPTParams) -> dict:\n    """Find load that maximizes output power."""\n    eta_values, pout_values, pin_values = load_sweep(load_values, frequency, k, params)\n    idx = int(np.nanargmax(pout_values))\n    return {\n        "RL_best": float(load_values[idx]),\n        "pout_best": float(pout_values[idx]),\n        "eta_at_best": float(eta_values[idx]),\n        "eta_at_best_percent": float(100.0 * eta_values[idx]),\n        "pin_at_best": float(pin_values[idx]),\n        "index": idx,\n    }\n\n\ndef find_best_load_constrained_eta(load_values, frequency: float, k: float, params: WPTParams,\n                                   pout_target_w: float) -> dict:\n    """Maximize efficiency subject to Pout >= pout_target_w.\n\n    Returns NaN values if no load in the search grid satisfies the power constraint.\n    """\n    eta_values, pout_values, pin_values = load_sweep(load_values, frequency, k, params)\n    feasible = np.where(pout_values >= pout_target_w)[0]\n    if feasible.size == 0:\n        return {\n            "RL_best": np.nan,\n            "eta_best": np.nan,\n            "eta_best_percent": np.nan,\n            "pout_at_best": np.nan,\n            "pin_at_best": np.nan,\n            "constraint_satisfied": False,\n            "index": -1,\n        }\n    local_idx = int(feasible[np.nanargmax(eta_values[feasible])])\n    return {\n        "RL_best": float(load_values[local_idx]),\n        "eta_best": float(eta_values[local_idx]),\n        "eta_best_percent": float(100.0 * eta_values[local_idx]),\n        "pout_at_best": float(pout_values[local_idx]),\n        "pin_at_best": float(pin_values[local_idx]),\n        "constraint_satisfied": True,\n        "index": local_idx,\n    }\n\n\ndef perturb_params(params: WPTParams, variation: float = 0.05, rng=None) -> WPTParams:\n    """Uniformly perturb R, L, C, and k_ref by +/- variation."""\n    if variation < 0:\n        raise ValueError("variation must be non-negative.")\n    if rng is None:\n        rng = np.random.default_rng()\n\n    def factor():\n        return float(1.0 + rng.uniform(-variation, variation))\n\n    return WPTParams(\n        R1=params.R1 * factor(),\n        R2=params.R2 * factor(),\n        L1=params.L1 * factor(),\n        L2=params.L2 * factor(),\n        C1=params.C1 * factor(),\n        C2=params.C2 * factor(),\n        Vs=params.Vs,\n        k_ref=float(np.clip(params.k_ref * factor(), 0.0, params.k_max)),\n        d_ref=params.d_ref,\n        n_decay=params.n_decay,\n        coil_radius=params.coil_radius,\n        k_max=params.k_max,\n    )\n'
RUN_WPT_PROJECT_CODE = '\n"""\nComplete corrected WPT robustness simulation pipeline.\n\nWhat this corrected version adds compared with the basic roadmap code:\n1. C2 and L2 detuning sweeps.\n2. Frequency tracking under combined misalignment + detuning.\n3. Load optimization using three objectives:\n   - maximum efficiency,\n   - maximum output power,\n   - maximum efficiency subject to Pout >= target.\n4. Figures and CSVs that report efficiency and output power together.\n5. Robustness thresholds of 50%, 60%, and 70%, with explicit interpretation.\n6. Corrected coupling model naming: k_ref is coupling at the reference distance d_ref.\n7. The final ZIP includes full source code in the code/ folder.\n"""\n\nfrom pathlib import Path\nimport csv\nimport json\nimport math\nimport os\nimport shutil\nimport sys\n\nimport numpy as np\nimport matplotlib.pyplot as plt\n\n# Make local model import work whether this script is run directly or by runpy.\nTHIS_FILE = Path(__file__).resolve()\nCODE_DIR = THIS_FILE.parent\nPROJECT_ROOT = CODE_DIR.parent\nif str(CODE_DIR) not in sys.path:\n    sys.path.insert(0, str(CODE_DIR))\n\nfrom advanced_wpt_model import (\n    WPTParams,\n    scaled_params,\n    resonance_frequency,\n    combined_coupling,\n    solve_circuit,\n    frequency_sweep,\n    load_sweep,\n    find_best_frequency,\n    find_best_load_max_eta,\n    find_best_load_max_pout,\n    find_best_load_constrained_eta,\n    perturb_params,\n    coupling_distance,\n)\n\nFIG_DIR = PROJECT_ROOT / "figures"\nDATA_DIR = PROJECT_ROOT / "data"\nREPORT_DIR = PROJECT_ROOT / "report"\nCODE_OUT_DIR = PROJECT_ROOT / "code"\nfor folder in [FIG_DIR, DATA_DIR, REPORT_DIR, CODE_OUT_DIR]:\n    folder.mkdir(parents=True, exist_ok=True)\n\nSHOW_PLOTS = False\nDPI = 300\n\n\ndef savefig(name: str):\n    path = FIG_DIR / name\n    plt.tight_layout()\n    plt.savefig(path, dpi=DPI, bbox_inches="tight")\n    if SHOW_PLOTS:\n        plt.show()\n    plt.close()\n    return path\n\n\ndef write_rows_csv(path: Path, header, rows):\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f)\n        writer.writerow(header)\n        writer.writerows(rows)\n\n\ndef safe_percent(value):\n    if value is None or (isinstance(value, float) and np.isnan(value)):\n        return np.nan\n    return 100.0 * float(value)\n\n\ndef make_equivalent_circuit_diagram():\n    """Create a simple equivalent-circuit schematic-like diagram using matplotlib."""\n    plt.figure(figsize=(10, 4))\n    ax = plt.gca()\n    ax.set_xlim(0, 10)\n    ax.set_ylim(0, 4)\n    ax.axis("off")\n\n    # Source and TX branch\n    ax.plot([0.6, 1.4], [2, 2], linewidth=2)\n    circle = plt.Circle((0.6, 2), 0.35, fill=False, linewidth=2)\n    ax.add_patch(circle)\n    ax.text(0.6, 1.25, "Vs", ha="center")\n    ax.text(1.8, 2.3, "R1, L1, C1", ha="center")\n    ax.plot([1.4, 3.4], [2, 2], linewidth=2)\n    ax.add_patch(plt.Rectangle((2.05, 1.75), 0.45, 0.5, fill=False, linewidth=2))\n    ax.text(2.28, 1.25, "TX coil", ha="center")\n\n    # Coupling indication\n    ax.annotate("Magnetic coupling M = k√(L1L2)", xy=(4.6, 2.2), xytext=(5.0, 3.2),\n                arrowprops=dict(arrowstyle="->", linewidth=1.5), ha="center")\n    ax.plot([3.8, 4.2], [1.5, 2.5], linewidth=2)\n    ax.plot([4.5, 4.9], [1.5, 2.5], linewidth=2)\n\n    # RX branch\n    ax.plot([5.2, 8.9], [2, 2], linewidth=2)\n    ax.add_patch(plt.Rectangle((5.9, 1.75), 0.45, 0.5, fill=False, linewidth=2))\n    ax.add_patch(plt.Rectangle((7.2, 1.65), 0.75, 0.7, fill=False, linewidth=2))\n    ax.text(6.12, 1.25, "RX coil", ha="center")\n    ax.text(7.58, 1.25, "RL", ha="center")\n    ax.text(6.7, 2.55, "R2, L2, C2", ha="center")\n    ax.text(5.0, 0.35, "Series-series resonant two-coil equivalent circuit", ha="center", fontsize=12)\n    savefig("week1_equivalent_circuit_diagram.png")\n\n\ndef week1_setup_and_parameters(base_params: WPTParams):\n    x = np.linspace(0, 10, 300)\n    y = np.sin(x) * np.exp(-0.1 * x)\n    plt.figure(figsize=(7, 4))\n    plt.plot(x, y)\n    plt.xlabel("x")\n    plt.ylabel("Damped sine wave")\n    plt.title("Week 1: Python Setup Check")\n    plt.grid(True, alpha=0.3)\n    savefig("week1_setup_check.png")\n\n    make_equivalent_circuit_diagram()\n\n    rows = [\n        ["R1_ohm", base_params.R1, "Transmitter coil resistance"],\n        ["R2_ohm", base_params.R2, "Receiver coil resistance"],\n        ["L1_H", base_params.L1, "Transmitter inductance"],\n        ["L2_H", base_params.L2, "Receiver inductance"],\n        ["C1_F", base_params.C1, "Transmitter compensation capacitance"],\n        ["C2_F", base_params.C2, "Receiver compensation capacitance"],\n        ["Vs_RMS_V", base_params.Vs, "RMS source voltage"],\n        ["k_ref", base_params.k_ref, "Coupling coefficient at d_ref, x=0, theta=0"],\n        ["d_ref_m", base_params.d_ref, "Reference distance where k(d_ref)=k_ref"],\n        ["n_decay", base_params.n_decay, "Distance decay exponent"],\n        ["coil_radius_m", base_params.coil_radius, "Coil radius used in lateral factor"],\n        ["k_max", base_params.k_max, "Upper clip for coupling coefficient"],\n        ["f0_nominal_Hz", resonance_frequency(base_params.L1, base_params.C1), "Nominal transmitter LC resonance"],\n    ]\n    write_rows_csv(DATA_DIR / "simulation_parameters.csv", ["Parameter", "Value", "Meaning"], rows)\n\n\ndef week3_detuning_sweeps(base_params: WPTParams):\n    """C2 and L2 detuning sweeps with peak frequency and power summaries."""\n    f0_nom = resonance_frequency(base_params.L1, base_params.C1)\n    freqs = np.linspace(0.55 * f0_nom, 1.45 * f0_nom, 900)\n    RL = 10.0\n    d = base_params.d_ref\n    x = 0.0\n    theta = 0.0\n\n    # C2 detuning response curves and summary\n    C2_scales = [0.80, 0.90, 1.00, 1.10, 1.20]\n    summary_rows = []\n    curve_rows = []\n    plt.figure(figsize=(8, 5))\n    for scale in C2_scales:\n        params = scaled_params(base_params, C2_scale=scale)\n        k = combined_coupling(d, x, theta, params)\n        eta, pout, pin = frequency_sweep(freqs, k, RL, params)\n        best_idx = int(np.argmax(eta))\n        fixed = solve_circuit(f0_nom, k, RL, params)\n        summary_rows.append([\n            scale,\n            params.C2,\n            resonance_frequency(params.L2, params.C2),\n            k,\n            fixed["eta_percent"],\n            fixed["Pout"],\n            float(100.0 * eta[best_idx]),\n            float(pout[best_idx]),\n            float(freqs[best_idx]),\n            float((freqs[best_idx] - f0_nom) / f0_nom * 100.0),\n        ])\n        plt.plot(freqs / 1000.0, eta * 100.0, label=f"C2 scale={scale:.2f}")\n        for f, e, p in zip(freqs, eta, pout):\n            curve_rows.append([scale, float(f), float(100.0 * e), float(p)])\n    plt.axvline(f0_nom / 1000.0, linestyle="--", linewidth=1.2, label="Nominal f0")\n    plt.xlabel("Frequency (kHz)")\n    plt.ylabel("Efficiency (%)")\n    plt.title("Week 3: C2 Detuning Frequency Response")\n    plt.grid(True, alpha=0.3)\n    plt.legend(fontsize=8)\n    savefig("week3_C2_detuning_frequency_response.png")\n    write_rows_csv(\n        DATA_DIR / "week3_C2_detuning_sweep_summary.csv",\n        ["C2_scale", "C2_F", "receiver_resonance_Hz", "k_eff", "eta_at_nominal_f_percent",\n         "Pout_at_nominal_f_W", "peak_eta_percent", "Pout_at_peak_eta_W", "best_frequency_Hz",\n         "best_frequency_shift_percent"],\n        summary_rows,\n    )\n    write_rows_csv(\n        DATA_DIR / "week3_C2_detuning_frequency_curves.csv",\n        ["C2_scale", "frequency_Hz", "eta_percent", "Pout_W"],\n        curve_rows,\n    )\n\n    # Plot peak eta and best frequency for C2 detuning\n    scales = np.array([r[0] for r in summary_rows])\n    peak_eta = np.array([r[6] for r in summary_rows])\n    best_freq_khz = np.array([r[8] for r in summary_rows]) / 1000.0\n    fig, ax1 = plt.subplots(figsize=(8, 5))\n    ax1.plot(scales, peak_eta, marker="o", label="Peak efficiency")\n    ax1.set_xlabel("C2 scale factor")\n    ax1.set_ylabel("Peak efficiency (%)")\n    ax1.grid(True, alpha=0.3)\n    ax2 = ax1.twinx()\n    ax2.plot(scales, best_freq_khz, marker="s", linestyle="--", label="Best frequency")\n    ax2.set_ylabel("Best frequency (kHz)")\n    plt.title("Week 3: C2 Detuning Shifts Best Frequency")\n    savefig("week3_C2_detuning_peak_eta_and_best_frequency.png")\n\n    # L2 detuning summary\n    L2_scales = [0.85, 0.95, 1.00, 1.05, 1.15]\n    L2_rows = []\n    plt.figure(figsize=(8, 5))\n    for scale in L2_scales:\n        params = scaled_params(base_params, L2_scale=scale)\n        k = combined_coupling(d, x, theta, params)\n        eta, pout, pin = frequency_sweep(freqs, k, RL, params)\n        best_idx = int(np.argmax(eta))\n        fixed = solve_circuit(f0_nom, k, RL, params)\n        L2_rows.append([\n            scale,\n            params.L2,\n            resonance_frequency(params.L2, params.C2),\n            k,\n            fixed["eta_percent"],\n            fixed["Pout"],\n            float(100.0 * eta[best_idx]),\n            float(pout[best_idx]),\n            float(freqs[best_idx]),\n            float((freqs[best_idx] - f0_nom) / f0_nom * 100.0),\n        ])\n        plt.plot(freqs / 1000.0, eta * 100.0, label=f"L2 scale={scale:.2f}")\n    plt.axvline(f0_nom / 1000.0, linestyle="--", linewidth=1.2, label="Nominal f0")\n    plt.xlabel("Frequency (kHz)")\n    plt.ylabel("Efficiency (%)")\n    plt.title("Week 3: L2 Detuning Frequency Response")\n    plt.grid(True, alpha=0.3)\n    plt.legend(fontsize=8)\n    savefig("week3_L2_detuning_frequency_response.png")\n    write_rows_csv(\n        DATA_DIR / "week3_L2_detuning_sweep_summary.csv",\n        ["L2_scale", "L2_H", "receiver_resonance_Hz", "k_eff", "eta_at_nominal_f_percent",\n         "Pout_at_nominal_f_W", "peak_eta_percent", "Pout_at_peak_eta_W", "best_frequency_Hz",\n         "best_frequency_shift_percent"],\n        L2_rows,\n    )\n\n\ndef week4_misalignment_maps(base_params: WPTParams):\n    f0_nom = resonance_frequency(base_params.L1, base_params.C1)\n    RL = 10.0\n\n    # Distance curve with power\n    distances = np.linspace(0.01, 0.20, 160)\n    rows = []\n    eta_distance, pout_distance, k_distance_values = [], [], []\n    for d in distances:\n        k = combined_coupling(d=d, x=0.0, theta_deg=0.0, params=base_params)\n        result = solve_circuit(f0_nom, k, RL, base_params)\n        rows.append([float(d), k, result["eta_percent"], result["Pout"], result["Pin"]])\n        eta_distance.append(result["eta_percent"])\n        pout_distance.append(result["Pout"])\n        k_distance_values.append(k)\n    write_rows_csv(DATA_DIR / "week4_distance_curve.csv",\n                   ["distance_m", "k_eff", "eta_percent", "Pout_W", "Pin_W"], rows)\n\n    fig, ax1 = plt.subplots(figsize=(8, 5))\n    ax1.plot(distances * 100.0, eta_distance, label="Efficiency")\n    ax1.set_xlabel("Distance (cm)")\n    ax1.set_ylabel("Efficiency (%)")\n    ax1.grid(True, alpha=0.3)\n    ax2 = ax1.twinx()\n    ax2.plot(distances * 100.0, pout_distance, linestyle="--", label="Output power")\n    ax2.set_ylabel("Output power (W)")\n    plt.title("Week 4: Distance Effect on Efficiency and Output Power")\n    savefig("week4_efficiency_and_power_vs_distance.png")\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(distances * 100.0, k_distance_values)\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Effective coupling k")\n    plt.title("Week 4: Corrected Distance Coupling Model, k(d_ref)=k_ref")\n    plt.grid(True, alpha=0.3)\n    savefig("week4_corrected_coupling_vs_distance.png")\n\n    # Lateral curve\n    shifts = np.linspace(0.0, 0.15, 160)\n    rows = []\n    eta_shift, pout_shift = [], []\n    for x in shifts:\n        k = combined_coupling(d=base_params.d_ref, x=x, theta_deg=0.0, params=base_params)\n        result = solve_circuit(f0_nom, k, RL, base_params)\n        rows.append([float(x), k, result["eta_percent"], result["Pout"], result["Pin"]])\n        eta_shift.append(result["eta_percent"])\n        pout_shift.append(result["Pout"])\n    write_rows_csv(DATA_DIR / "week4_lateral_curve.csv",\n                   ["lateral_shift_m", "k_eff", "eta_percent", "Pout_W", "Pin_W"], rows)\n    fig, ax1 = plt.subplots(figsize=(8, 5))\n    ax1.plot(shifts * 100.0, eta_shift, label="Efficiency")\n    ax1.set_xlabel("Lateral shift (cm)")\n    ax1.set_ylabel("Efficiency (%)")\n    ax1.grid(True, alpha=0.3)\n    ax2 = ax1.twinx()\n    ax2.plot(shifts * 100.0, pout_shift, linestyle="--", label="Output power")\n    ax2.set_ylabel("Output power (W)")\n    plt.title("Week 4: Lateral Misalignment Effect")\n    savefig("week4_efficiency_and_power_vs_lateral_shift.png")\n\n    # Tilt curve\n    angles = np.linspace(0.0, 80.0, 160)\n    rows = []\n    eta_tilt, pout_tilt = [], []\n    for theta in angles:\n        k = combined_coupling(d=base_params.d_ref, x=0.0, theta_deg=float(theta), params=base_params)\n        result = solve_circuit(f0_nom, k, RL, base_params)\n        rows.append([float(theta), k, result["eta_percent"], result["Pout"], result["Pin"]])\n        eta_tilt.append(result["eta_percent"])\n        pout_tilt.append(result["Pout"])\n    write_rows_csv(DATA_DIR / "week4_tilt_curve.csv",\n                   ["tilt_deg", "k_eff", "eta_percent", "Pout_W", "Pin_W"], rows)\n    fig, ax1 = plt.subplots(figsize=(8, 5))\n    ax1.plot(angles, eta_tilt, label="Efficiency")\n    ax1.set_xlabel("Tilt angle (degrees)")\n    ax1.set_ylabel("Efficiency (%)")\n    ax1.grid(True, alpha=0.3)\n    ax2 = ax1.twinx()\n    ax2.plot(angles, pout_tilt, linestyle="--", label="Output power")\n    ax2.set_ylabel("Output power (W)")\n    plt.title("Week 4: Angular Tilt Effect")\n    savefig("week4_efficiency_and_power_vs_tilt.png")\n\n    # Heatmaps for combined distance + lateral shift\n    d_grid = np.linspace(0.01, 0.20, 90)\n    x_grid = np.linspace(0.0, 0.15, 90)\n    eta_map = np.zeros((len(x_grid), len(d_grid)))\n    pout_map = np.zeros_like(eta_map)\n    for i, x in enumerate(x_grid):\n        for j, d in enumerate(d_grid):\n            k = combined_coupling(d=d, x=x, theta_deg=15.0, params=base_params)\n            result = solve_circuit(f0_nom, k, RL, base_params)\n            eta_map[i, j] = result["eta_percent"]\n            pout_map[i, j] = result["Pout"]\n\n    plt.figure(figsize=(8, 5))\n    im = plt.imshow(eta_map, origin="lower", aspect="auto",\n                    extent=[d_grid.min() * 100, d_grid.max() * 100, x_grid.min() * 100, x_grid.max() * 100])\n    plt.colorbar(im, label="Efficiency (%)")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Lateral shift (cm)")\n    plt.title("Week 4: Efficiency Heatmap, Distance + Lateral Shift, Tilt=15°")\n    savefig("week4_efficiency_heatmap_distance_lateral.png")\n\n    plt.figure(figsize=(8, 5))\n    im = plt.imshow(pout_map, origin="lower", aspect="auto",\n                    extent=[d_grid.min() * 100, d_grid.max() * 100, x_grid.min() * 100, x_grid.max() * 100])\n    plt.colorbar(im, label="Output power (W)")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Lateral shift (cm)")\n    plt.title("Week 4: Output Power Heatmap, Distance + Lateral Shift, Tilt=15°")\n    savefig("week4_output_power_heatmap_distance_lateral.png")\n\n    heatmap_rows = []\n    for i, x in enumerate(x_grid):\n        for j, d in enumerate(d_grid):\n            heatmap_rows.append([float(d), float(x), 15.0, float(eta_map[i, j]), float(pout_map[i, j])])\n    write_rows_csv(DATA_DIR / "week4_distance_lateral_heatmap_data.csv",\n                   ["distance_m", "lateral_shift_m", "tilt_deg", "eta_percent", "Pout_W"], heatmap_rows)\n\n\ndef week5_tracking_combined_misalignment_detuning(base_params: WPTParams):\n    """Frequency tracking is tested under combined misalignment + C2/L2 detuning."""\n    f0_nom = resonance_frequency(base_params.L1, base_params.C1)\n    params_detuned = scaled_params(base_params, C2_scale=0.80, L2_scale=1.05)\n    freq_grid = np.linspace(0.55 * f0_nom, 1.45 * f0_nom, 900)\n    RL = 10.0\n    x = 0.04\n    theta = 20.0\n    distances = np.linspace(0.01, 0.20, 100)\n    rows = []\n\n    for d in distances:\n        k = combined_coupling(d=d, x=x, theta_deg=theta, params=params_detuned)\n        fixed = solve_circuit(f0_nom, k, RL, params_detuned)\n        tracked = find_best_frequency(freq_grid, k, RL, params_detuned)\n        rows.append([\n            float(d), x, theta, 0.80, 1.05, k,\n            fixed["eta_percent"], fixed["Pout"], fixed["Pin"],\n            tracked["eta_best_percent"], tracked["pout_at_best"], tracked["pin_at_best"], tracked["f_best"],\n            tracked["eta_best_percent"] - fixed["eta_percent"],\n            tracked["pout_at_best"] - fixed["Pout"],\n        ])\n\n    header = ["distance_m", "lateral_shift_m", "tilt_deg", "C2_scale", "L2_scale", "k_eff",\n              "eta_fixed_percent", "Pout_fixed_W", "Pin_fixed_W",\n              "eta_tracked_percent", "Pout_tracked_W", "Pin_tracked_W", "best_frequency_Hz",\n              "tracking_eta_gain_percentage_points", "tracking_Pout_change_W"]\n    write_rows_csv(DATA_DIR / "week5_tracking_under_misalignment_and_detuning.csv", header, rows)\n\n    arr = np.asarray(rows, dtype=float)\n    d_cm = arr[:, 0] * 100.0\n    eta_fixed = arr[:, 6]\n    pout_fixed = arr[:, 7]\n    eta_tracked = arr[:, 9]\n    pout_tracked = arr[:, 10]\n    best_freq_khz = arr[:, 12] / 1000.0\n    gain = arr[:, 13]\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(d_cm, eta_fixed, label="Fixed frequency")\n    plt.plot(d_cm, eta_tracked, label="Tracked frequency")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Efficiency (%)")\n    plt.title("Week 5: Frequency Tracking under Misalignment + Detuning")\n    plt.grid(True, alpha=0.3)\n    plt.legend()\n    savefig("week5_fixed_vs_tracked_efficiency_detuned.png")\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(d_cm, pout_fixed, label="Fixed frequency")\n    plt.plot(d_cm, pout_tracked, label="Tracked frequency")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Output power (W)")\n    plt.title("Week 5: Output Power Must Be Reported with Tracking")\n    plt.grid(True, alpha=0.3)\n    plt.legend()\n    savefig("week5_fixed_vs_tracked_output_power_detuned.png")\n\n    fig, ax1 = plt.subplots(figsize=(8, 5))\n    ax1.plot(d_cm, eta_fixed, label="η fixed")\n    ax1.plot(d_cm, eta_tracked, label="η tracked")\n    ax1.set_xlabel("Distance (cm)")\n    ax1.set_ylabel("Efficiency (%)")\n    ax1.grid(True, alpha=0.3)\n    ax2 = ax1.twinx()\n    ax2.plot(d_cm, pout_fixed, linestyle="--", label="Pout fixed")\n    ax2.plot(d_cm, pout_tracked, linestyle=":", label="Pout tracked")\n    ax2.set_ylabel("Output power (W)")\n    plt.title("Week 5: Efficiency and Power Together")\n    savefig("week5_tracking_efficiency_and_power_together.png")\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(d_cm, best_freq_khz)\n    plt.axhline(f0_nom / 1000.0, linestyle="--", linewidth=1.2, label="Nominal f0")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Best frequency (kHz)")\n    plt.title("Week 5: Best Frequency under Combined Non-Ideality")\n    plt.grid(True, alpha=0.3)\n    plt.legend()\n    savefig("week5_best_frequency_vs_distance_detuned.png")\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(d_cm, gain)\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Tracking gain (percentage points)")\n    plt.title("Week 5: Tracking Efficiency Gain")\n    plt.grid(True, alpha=0.3)\n    savefig("week5_tracking_gain_vs_distance_detuned.png")\n\n    # Additional 2D tracking map: this prevents the misleading interpretation that\n    # tracking is only a distance curve. The best frequency is expected to vary\n    # primarily with detuning, while distance controls coupling strength.\n    d_grid = np.linspace(0.01, 0.20, 25)\n    c2_grid = np.linspace(0.75, 1.25, 25)\n    freq_grid_map = np.linspace(0.55 * f0_nom, 1.45 * f0_nom, 350)\n    gain_map = np.zeros((len(c2_grid), len(d_grid)))\n    best_freq_map = np.zeros_like(gain_map)\n    eta_fixed_map = np.zeros_like(gain_map)\n    eta_tracked_map = np.zeros_like(gain_map)\n    heat_rows = []\n    for i, c2_scale in enumerate(c2_grid):\n        params_case = scaled_params(base_params, C2_scale=float(c2_scale), L2_scale=1.05)\n        for j, dval in enumerate(d_grid):\n            kval = combined_coupling(d=float(dval), x=x, theta_deg=theta, params=params_case)\n            fixed_case = solve_circuit(f0_nom, kval, RL, params_case)\n            tracked_case = find_best_frequency(freq_grid_map, kval, RL, params_case)\n            eta_fixed_map[i, j] = fixed_case["eta_percent"]\n            eta_tracked_map[i, j] = tracked_case["eta_best_percent"]\n            gain_map[i, j] = tracked_case["eta_best_percent"] - fixed_case["eta_percent"]\n            best_freq_map[i, j] = tracked_case["f_best"] / 1000.0\n            heat_rows.append([float(dval), float(c2_scale), 1.05, x, theta, kval,\n                              fixed_case["eta_percent"], tracked_case["eta_best_percent"],\n                              gain_map[i, j], tracked_case["f_best"],\n                              fixed_case["Pout"], tracked_case["pout_at_best"]])\n    write_rows_csv(DATA_DIR / "week5_tracking_heatmap_distance_C2.csv",\n                   ["distance_m", "C2_scale", "L2_scale", "lateral_shift_m", "tilt_deg", "k_eff",\n                    "eta_fixed_percent", "eta_tracked_percent", "tracking_gain_pp", "best_frequency_Hz",\n                    "Pout_fixed_W", "Pout_tracked_W"], heat_rows)\n\n    plt.figure(figsize=(8, 5))\n    im = plt.imshow(gain_map, origin="lower", aspect="auto",\n                    extent=[d_grid.min() * 100, d_grid.max() * 100, c2_grid.min(), c2_grid.max()])\n    plt.colorbar(im, label="Tracking gain (percentage points)")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("C2 scale")\n    plt.title("Week 5: Tracking Gain Map under Distance + C2 Detuning")\n    savefig("week5_tracking_gain_heatmap_distance_C2.png")\n\n    plt.figure(figsize=(8, 5))\n    im = plt.imshow(best_freq_map, origin="lower", aspect="auto",\n                    extent=[d_grid.min() * 100, d_grid.max() * 100, c2_grid.min(), c2_grid.max()])\n    plt.colorbar(im, label="Best frequency (kHz)")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("C2 scale")\n    plt.title("Week 5: Best Frequency Map under Distance + C2 Detuning")\n    savefig("week5_best_frequency_heatmap_distance_C2.png")\n\n\ndef week6_load_optimization_multiobjective(base_params: WPTParams):\n    f0_nom = resonance_frequency(base_params.L1, base_params.C1)\n    params_detuned = scaled_params(base_params, C2_scale=0.80, L2_scale=1.05)\n    load_grid = np.linspace(1.0, 200.0, 500)\n    distances = np.linspace(0.01, 0.20, 100)\n    x = 0.04\n    theta = 20.0\n    RL_nominal = 10.0\n    pout_target_w = 10.0\n    rows = []\n\n    for d in distances:\n        k = combined_coupling(d=d, x=x, theta_deg=theta, params=params_detuned)\n        nominal = solve_circuit(f0_nom, k, RL_nominal, params_detuned)\n        max_eta = find_best_load_max_eta(load_grid, f0_nom, k, params_detuned)\n        max_pout = find_best_load_max_pout(load_grid, f0_nom, k, params_detuned)\n        constrained = find_best_load_constrained_eta(load_grid, f0_nom, k, params_detuned, pout_target_w)\n        rows.append([\n            float(d), x, theta, 0.80, 1.05, k, RL_nominal,\n            nominal["eta_percent"], nominal["Pout"], nominal["Pin"],\n            max_eta["RL_best"], max_eta["eta_best_percent"], max_eta["pout_at_best"], max_eta["pin_at_best"],\n            max_pout["RL_best"], max_pout["eta_at_best_percent"], max_pout["pout_best"], max_pout["pin_at_best"],\n            pout_target_w, constrained["RL_best"], constrained["eta_best_percent"], constrained["pout_at_best"],\n            constrained["pin_at_best"], constrained["constraint_satisfied"],\n        ])\n\n    header = ["distance_m", "lateral_shift_m", "tilt_deg", "C2_scale", "L2_scale", "k_eff", "RL_nominal_ohm",\n              "eta_nominal_percent", "Pout_nominal_W", "Pin_nominal_W",\n              "RL_max_eta_ohm", "eta_max_eta_percent", "Pout_at_max_eta_W", "Pin_at_max_eta_W",\n              "RL_max_pout_ohm", "eta_at_max_pout_percent", "Pout_max_W", "Pin_at_max_pout_W",\n              "Pout_target_W", "RL_constrained_ohm", "eta_constrained_percent", "Pout_constrained_W",\n              "Pin_constrained_W", "constraint_satisfied"]\n    write_rows_csv(DATA_DIR / "week6_multiobjective_load_optimization.csv", header, rows)\n\n    arr = np.asarray([[float(v) if isinstance(v, (int, float, np.floating, bool)) else np.nan for v in row] for row in rows], dtype=float)\n    d_cm = arr[:, 0] * 100.0\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(d_cm, arr[:, 7], label="Nominal load")\n    plt.plot(d_cm, arr[:, 11], label="Max efficiency")\n    plt.plot(d_cm, arr[:, 15], label="Max output power objective")\n    plt.plot(d_cm, arr[:, 20], label="Constrained η, Pout ≥ 10 W")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Efficiency (%)")\n    plt.title("Week 6: Load Objectives Compared by Efficiency")\n    plt.grid(True, alpha=0.3)\n    plt.legend(fontsize=8)\n    savefig("week6_load_objectives_efficiency_vs_distance.png")\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(d_cm, arr[:, 8], label="Nominal load")\n    plt.plot(d_cm, arr[:, 12], label="At max efficiency")\n    plt.plot(d_cm, arr[:, 16], label="Maximum output power")\n    plt.plot(d_cm, arr[:, 21], label="Constrained η, Pout ≥ 10 W")\n    plt.axhline(pout_target_w, linestyle="--", linewidth=1.2, label="Power target")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Output power (W)")\n    plt.title("Week 6: Load Objectives Compared by Output Power")\n    plt.grid(True, alpha=0.3)\n    plt.legend(fontsize=8)\n    savefig("week6_load_objectives_output_power_vs_distance.png")\n\n    plt.figure(figsize=(8, 5))\n    plt.plot(d_cm, arr[:, 10], label="Load for max η")\n    plt.plot(d_cm, arr[:, 14], label="Load for max Pout")\n    plt.plot(d_cm, arr[:, 19], label="Load for constrained η")\n    plt.xlabel("Distance (cm)")\n    plt.ylabel("Best load resistance (Ω)")\n    plt.title("Week 6: Best Load Depends on Objective")\n    plt.grid(True, alpha=0.3)\n    plt.legend(fontsize=8)\n    savefig("week6_best_loads_for_different_objectives.png")\n\n    # Detailed tradeoff at one representative distance\n    d_example = 0.08\n    k = combined_coupling(d=d_example, x=x, theta_deg=theta, params=params_detuned)\n    eta, pout, pin = load_sweep(load_grid, f0_nom, k, params_detuned)\n    trade_rows = [[float(RL), float(e * 100.0), float(p), float(pi)] for RL, e, p, pi in zip(load_grid, eta, pout, pin)]\n    write_rows_csv(DATA_DIR / "week6_eta_power_tradeoff_example_d_8cm.csv",\n                   ["RL_ohm", "eta_percent", "Pout_W", "Pin_W"], trade_rows)\n    fig, ax1 = plt.subplots(figsize=(8, 5))\n    ax1.plot(load_grid, eta * 100.0, label="Efficiency")\n    ax1.set_xlabel("Load resistance RL (Ω)")\n    ax1.set_ylabel("Efficiency (%)")\n    ax1.grid(True, alpha=0.3)\n    ax2 = ax1.twinx()\n    ax2.plot(load_grid, pout, linestyle="--", label="Output power")\n    ax2.set_ylabel("Output power (W)")\n    plt.title("Week 6: Efficiency-Power Tradeoff at d=8 cm")\n    savefig("week6_efficiency_power_tradeoff_example.png")\n\n\ndef week7_robustness_thresholds_and_sensitivity(base_params: WPTParams):\n    f0_nom = resonance_frequency(base_params.L1, base_params.C1)\n    params_detuned = scaled_params(base_params, C2_scale=0.80, L2_scale=1.05)\n    RL = 10.0\n    d = 0.08\n    x = 0.03\n    theta = 15.0\n    variation = 0.05\n    num_samples = 2000\n    rng = np.random.default_rng(42)\n    eta_samples = []\n    pout_samples = []\n\n    for _ in range(num_samples):\n        sampled = perturb_params(params_detuned, variation=variation, rng=rng)\n        k_sample = combined_coupling(d=d, x=x, theta_deg=theta, params=sampled)\n        result = solve_circuit(f0_nom, k_sample, RL, sampled)\n        eta_samples.append(result["eta_percent"])\n        pout_samples.append(result["Pout"])\n\n    eta_samples = np.asarray(eta_samples)\n    pout_samples = np.asarray(pout_samples)\n    thresholds = [50.0, 60.0, 70.0]\n    summary_rows = [\n        ["Number of samples", num_samples],\n        ["Parameter variation range", f"±{variation * 100:.1f}%"],\n        ["Mean efficiency (%)", float(np.mean(eta_samples))],\n        ["Std efficiency (%)", float(np.std(eta_samples, ddof=1))],\n        ["Minimum efficiency (%)", float(np.min(eta_samples))],\n        ["Maximum efficiency (%)", float(np.max(eta_samples))],\n        ["10th percentile efficiency (%)", float(np.percentile(eta_samples, 10))],\n        ["90th percentile efficiency (%)", float(np.percentile(eta_samples, 90))],\n        ["Mean output power (W)", float(np.mean(pout_samples))],\n        ["Std output power (W)", float(np.std(pout_samples, ddof=1))],\n    ]\n    for thr in thresholds:\n        summary_rows.append([f"Probability eta >= {thr:.0f}%", float(np.mean(eta_samples >= thr))])\n    write_rows_csv(DATA_DIR / "week7_monte_carlo_summary.csv", ["Metric", "Value"], summary_rows)\n    write_rows_csv(DATA_DIR / "week7_monte_carlo_raw_samples.csv",\n                   ["sample_index", "eta_percent", "Pout_W"],\n                   [[i, float(e), float(p)] for i, (e, p) in enumerate(zip(eta_samples, pout_samples))])\n\n    threshold_rows = [\n        [50, "Lenient/minimum acceptable", "Below 50%, the equivalent-circuit transfer is weak for this simulation scale; useful for identifying severe failure regions."],\n        [60, "Moderate robustness", "Requires a clear margin above the minimum and is more realistic for comparing acceptable operating regions."],\n        [70, "Strict/high-efficiency", "Represents strong operation close to the high-efficiency region; many misaligned cases may fail this threshold."],\n    ]\n    write_rows_csv(DATA_DIR / "week7_robustness_threshold_justification.csv",\n                   ["threshold_percent", "interpretation", "reason"], threshold_rows)\n\n    plt.figure(figsize=(8, 5))\n    plt.hist(eta_samples, bins=35, edgecolor="black")\n    for thr in thresholds:\n        plt.axvline(thr, linestyle="--", linewidth=1.2, label=f"{thr:.0f}% threshold")\n    plt.xlabel("Efficiency (%)")\n    plt.ylabel("Number of samples")\n    plt.title("Week 7: Monte Carlo Efficiency Distribution with Thresholds")\n    plt.grid(True, alpha=0.3)\n    plt.legend(fontsize=8)\n    savefig("week7_monte_carlo_efficiency_histogram_thresholds.png")\n\n    # Sensitivity ranking\n    parameter_names = ["R1", "R2", "L1", "L2", "C1", "C2", "k_ref"]\n    sensitivity_rows = []\n    for name in parameter_names:\n        params_plus = scaled_params(params_detuned)\n        params_minus = scaled_params(params_detuned)\n        setattr(params_plus, name, getattr(params_plus, name) * 1.05)\n        setattr(params_minus, name, getattr(params_minus, name) * 0.95)\n        k_plus = combined_coupling(d=d, x=x, theta_deg=theta, params=params_plus)\n        k_minus = combined_coupling(d=d, x=x, theta_deg=theta, params=params_minus)\n        eta_plus = solve_circuit(f0_nom, k_plus, RL, params_plus)["eta_percent"]\n        eta_minus = solve_circuit(f0_nom, k_minus, RL, params_minus)["eta_percent"]\n        pout_plus = solve_circuit(f0_nom, k_plus, RL, params_plus)["Pout"]\n        pout_minus = solve_circuit(f0_nom, k_minus, RL, params_minus)["Pout"]\n        sensitivity_eta_per_1pct = abs(eta_plus - eta_minus) / 10.0\n        sensitivity_pout_per_1pct = abs(pout_plus - pout_minus) / 10.0\n        sensitivity_rows.append([name, sensitivity_eta_per_1pct, eta_minus, eta_plus,\n                                 sensitivity_pout_per_1pct, pout_minus, pout_plus])\n    sensitivity_rows.sort(key=lambda row: row[1], reverse=True)\n    write_rows_csv(DATA_DIR / "week7_sensitivity_ranking.csv",\n                   ["Parameter", "eta_sensitivity_pp_per_1_percent_change", "eta_at_minus_5_percent",\n                    "eta_at_plus_5_percent", "Pout_sensitivity_W_per_1_percent_change",\n                    "Pout_at_minus_5_percent", "Pout_at_plus_5_percent"], sensitivity_rows)\n\n    plt.figure(figsize=(8, 5))\n    plt.bar([r[0] for r in sensitivity_rows], [r[1] for r in sensitivity_rows])\n    plt.xlabel("Parameter")\n    plt.ylabel("η sensitivity, percentage points per 1% change")\n    plt.title("Week 7: One-at-a-Time Sensitivity Ranking")\n    plt.grid(True, axis="y", alpha=0.3)\n    savefig("week7_sensitivity_ranking_efficiency.png")\n\n    # Robustness maps for 50, 60, 70 percent thresholds\n    d_grid = np.linspace(0.01, 0.20, 100)\n    x_grid = np.linspace(0.0, 0.15, 100)\n    eta_map = np.zeros((len(x_grid), len(d_grid)))\n    for i, xval in enumerate(x_grid):\n        for j, dval in enumerate(d_grid):\n            kval = combined_coupling(d=dval, x=xval, theta_deg=15.0, params=params_detuned)\n            eta_map[i, j] = solve_circuit(f0_nom, kval, RL, params_detuned)["eta_percent"]\n\n    map_rows = []\n    for i, xval in enumerate(x_grid):\n        for j, dval in enumerate(d_grid):\n            map_rows.append([float(dval), float(xval), 15.0, float(eta_map[i, j]),\n                             int(eta_map[i, j] >= 50), int(eta_map[i, j] >= 60), int(eta_map[i, j] >= 70)])\n    write_rows_csv(DATA_DIR / "week7_threshold_robustness_map_data.csv",\n                   ["distance_m", "lateral_shift_m", "tilt_deg", "eta_percent",\n                    "robust_50", "robust_60", "robust_70"], map_rows)\n\n    region_rows = []\n    for thr in thresholds:\n        robust = eta_map >= thr\n        robust_fraction = float(np.mean(robust))\n        region_rows.append([thr, robust_fraction, robust_fraction * 100.0])\n        plt.figure(figsize=(8, 5))\n        im = plt.imshow(robust.astype(float), origin="lower", aspect="auto",\n                        extent=[d_grid.min() * 100, d_grid.max() * 100, x_grid.min() * 100, x_grid.max() * 100],\n                        vmin=0, vmax=1)\n        plt.colorbar(im, label=f"η ≥ {thr:.0f}% ?")\n        plt.xlabel("Distance (cm)")\n        plt.ylabel("Lateral shift (cm)")\n        plt.title(f"Week 7: Robustness Map at {thr:.0f}% Efficiency Threshold")\n        savefig(f"week7_robustness_map_threshold_{int(thr)}.png")\n    write_rows_csv(DATA_DIR / "week7_robust_operating_area_by_threshold.csv",\n                   ["threshold_percent", "fraction_of_grid_robust", "percent_of_grid_robust"], region_rows)\n\n\ndef week8_final_comparison_and_report(base_params: WPTParams):\n    f0_nom = resonance_frequency(base_params.L1, base_params.C1)\n    frequency_grid = np.linspace(0.55 * f0_nom, 1.45 * f0_nom, 900)\n    load_grid = np.linspace(1.0, 200.0, 500)\n    RL_nominal = 10.0\n    pout_target_w = 10.0\n\n    cases = [\n        ["Ideal aligned tuned", 0.05, 0.00, 0.0, 1.00, 1.00],\n        ["C2 detuned only", 0.05, 0.00, 0.0, 0.80, 1.00],\n        ["L2 detuned only", 0.05, 0.00, 0.0, 1.00, 1.05],\n        ["Distance + detuning", 0.10, 0.00, 0.0, 0.80, 1.05],\n        ["Lateral shift + detuning", 0.05, 0.06, 0.0, 0.80, 1.05],\n        ["Tilt + detuning", 0.05, 0.00, 35.0, 0.80, 1.05],\n        ["Combined non-ideal", 0.10, 0.06, 30.0, 0.80, 1.05],\n    ]\n    rows = []\n    for name, d, x, theta, c2s, l2s in cases:\n        params_case = scaled_params(base_params, C2_scale=c2s, L2_scale=l2s)\n        k = combined_coupling(d=d, x=x, theta_deg=theta, params=params_case)\n        fixed = solve_circuit(f0_nom, k, RL_nominal, params_case)\n        tracked = find_best_frequency(frequency_grid, k, RL_nominal, params_case)\n        max_eta = find_best_load_max_eta(load_grid, f0_nom, k, params_case)\n        max_pout = find_best_load_max_pout(load_grid, f0_nom, k, params_case)\n        constrained = find_best_load_constrained_eta(load_grid, f0_nom, k, params_case, pout_target_w)\n        rows.append([\n            name, d, x, theta, c2s, l2s, k,\n            fixed["eta_percent"], fixed["Pout"],\n            tracked["eta_best_percent"], tracked["pout_at_best"], tracked["f_best"],\n            tracked["eta_best_percent"] - fixed["eta_percent"],\n            tracked["pout_at_best"] - fixed["Pout"],\n            max_eta["RL_best"], max_eta["eta_best_percent"], max_eta["pout_at_best"],\n            max_pout["RL_best"], max_pout["eta_at_best_percent"], max_pout["pout_best"],\n            pout_target_w, constrained["RL_best"], constrained["eta_best_percent"], constrained["pout_at_best"],\n            constrained["constraint_satisfied"],\n        ])\n    header = ["Case", "distance_m", "lateral_shift_m", "tilt_deg", "C2_scale", "L2_scale", "k_eff",\n              "fixed_eta_percent", "fixed_Pout_W",\n              "tracked_eta_percent", "tracked_Pout_W", "tracked_best_frequency_Hz",\n              "tracking_eta_gain_pp", "tracking_Pout_change_W",\n              "RL_max_eta_ohm", "max_eta_percent", "Pout_at_max_eta_W",\n              "RL_max_pout_ohm", "eta_at_max_pout_percent", "max_Pout_W",\n              "Pout_target_W", "RL_constrained_ohm", "constrained_eta_percent", "constrained_Pout_W",\n              "constraint_satisfied"]\n    write_rows_csv(DATA_DIR / "final_comparison_table.csv", header, rows)\n\n    # Read threshold files for report summaries.\n    mc_summary_path = DATA_DIR / "week7_monte_carlo_summary.csv"\n    threshold_area_path = DATA_DIR / "week7_robust_operating_area_by_threshold.csv"\n\n    report_lines = []\n    report_lines.append("Corrected WPT Robustness Project: Result Interpretation")\n    report_lines.append("=" * 68)\n    report_lines.append("")\n    report_lines.append("Model correction:")\n    report_lines.append("- k_ref is now explicitly defined as the coupling coefficient at d_ref when x=0 and theta=0.")\n    report_lines.append("- The distance model is normalized so that k(d_ref)=k_ref. This avoids the earlier ambiguity where k0 looked like a reference coupling but actually became half of that value at d0.")\n    report_lines.append("")\n    report_lines.append("Scientific corrections added:")\n    report_lines.append("1. C2 and L2 detuning sweeps are included.")\n    report_lines.append("2. Frequency tracking is tested under combined misalignment plus detuning, not only misalignment.")\n    report_lines.append("3. Load optimization is reported for maximum efficiency, maximum output power, and constrained efficiency with Pout >= 10 W.")\n    report_lines.append("4. Efficiency and output power are both plotted and saved in CSV files.")\n    report_lines.append("5. Robustness thresholds of 50%, 60%, and 70% are explicitly justified and reported.")\n    report_lines.append("")\n    report_lines.append("Important interpretation:")\n    report_lines.append("- Frequency tracking is meaningful mainly when detuning exists. If only coupling changes while both circuits remain perfectly tuned, tracking may produce only marginal gains.")\n    report_lines.append("- Efficiency-only optimization can be misleading because it may choose a load that gives high efficiency but low delivered power.")\n    report_lines.append("- Output-power-aware and constrained optimization are therefore necessary for honest interpretation.")\n    report_lines.append("- A 50% robustness threshold is lenient; 60% is moderate; 70% is strict. A design that passes 50% but fails 70% should not be called strongly robust.")\n    report_lines.append("")\n    report_lines.append("Key output files:")\n    report_lines.append("- data/week3_C2_detuning_sweep_summary.csv")\n    report_lines.append("- data/week3_L2_detuning_sweep_summary.csv")\n    report_lines.append("- data/week5_tracking_under_misalignment_and_detuning.csv")\n    report_lines.append("- data/week6_multiobjective_load_optimization.csv")\n    report_lines.append("- data/week7_robustness_threshold_justification.csv")\n    report_lines.append("- data/week7_robust_operating_area_by_threshold.csv")\n    report_lines.append("- data/final_comparison_table.csv")\n    report_lines.append("")\n    report_lines.append("Recommended final conclusion wording:")\n    report_lines.append("The equivalent-circuit simulations show that misalignment, detuning, and load variation can jointly degrade two-coil resonant WPT performance. Frequency tracking recovers efficiency mainly under detuned conditions, while load optimization has a stronger but objective-dependent effect. Because maximum-efficiency load selection can reduce useful delivered power, the final design assessment should report both efficiency and output power and should include a constrained objective such as maximizing efficiency subject to a minimum output-power requirement. Robustness depends strongly on whether the acceptable efficiency threshold is taken as 50%, 60%, or 70%.")\n\n    (REPORT_DIR / "corrected_result_interpretation.txt").write_text("\\n".join(report_lines), encoding="utf-8")\n\n    # JSON manifest\n    manifest = {\n        "project": "Robustness-aware two-coil resonant WPT simulation",\n        "version": "corrected_v2",\n        "contains_full_source_code": True,\n        "important_corrections": [\n            "C2 and L2 detuning sweeps",\n            "frequency tracking under misalignment plus detuning",\n            "multi-objective load optimization",\n            "efficiency and power reported together",\n            "50/60/70 percent robustness thresholds with justification",\n            "normalized k(d_ref)=k_ref coupling model",\n            "source code included inside ZIP",\n        ],\n    }\n    (REPORT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n\n\ndef zip_project():\n    # Keep the distributed ZIP clean and reproducible: include source .py files,\n    # but exclude Python cache artifacts created during import.\n    shutil.rmtree(CODE_OUT_DIR / "__pycache__", ignore_errors=True)\n    zip_base = PROJECT_ROOT.parent / "WPT_Robustness_Project_Corrected_Results"\n    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=PROJECT_ROOT)\n    return Path(zip_path)\n\n\ndef main():\n    base_params = WPTParams()\n    print("Running corrected WPT robustness simulation...")\n    print(f"Project root: {PROJECT_ROOT}")\n\n    week1_setup_and_parameters(base_params)\n    week3_detuning_sweeps(base_params)\n    week4_misalignment_maps(base_params)\n    week5_tracking_combined_misalignment_detuning(base_params)\n    week6_load_optimization_multiobjective(base_params)\n    week7_robustness_thresholds_and_sensitivity(base_params)\n    week8_final_comparison_and_report(base_params)\n\n    zip_path = zip_project()\n    print("\\nCompleted successfully.")\n    print(f"Figures saved to: {FIG_DIR}")\n    print(f"CSV data saved to: {DATA_DIR}")\n    print(f"Report saved to: {REPORT_DIR}")\n    print(f"ZIP saved to: {zip_path}")\n    print("The ZIP includes full source code in the code/ folder.")\n\n    # Auto-download only when running in Google Colab.\n    try:\n        from google.colab import files  # type: ignore\n        files.download(str(zip_path))\n    except Exception:\n        print("Not running inside Google Colab, so automatic browser download was skipped.")\n        print("Open or download the ZIP manually from the printed path.")\n\n\nif __name__ == "__main__":\n    main()\n'

(CODE_DIR / "advanced_wpt_model.py").write_text(ADVANCED_WPT_MODEL_CODE, encoding="utf-8")
(CODE_DIR / "run_wpt_project.py").write_text(RUN_WPT_PROJECT_CODE, encoding="utf-8")
(CODE_DIR / "README.txt").write_text(
    "This folder contains the full source code used to generate all corrected WPT results.\n"
    "advanced_wpt_model.py: reusable equivalent-circuit model.\n"
    "run_wpt_project.py: full week-wise simulation pipeline.\n"
    "The final ZIP intentionally includes this code folder for reproducibility.\n",
    encoding="utf-8",
)

if str(CODE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(CODE_DIR.resolve()))

runpy.run_path(str((CODE_DIR / "run_wpt_project.py").resolve()), run_name="__main__")


Running corrected WPT robustness simulation...
Project root: /content/WPT_Robustness_Project_Corrected

Completed successfully.
Figures saved to: /content/WPT_Robustness_Project_Corrected/figures
CSV data saved to: /content/WPT_Robustness_Project_Corrected/data
Report saved to: /content/WPT_Robustness_Project_Corrected/report
ZIP saved to: /content/WPT_Robustness_Project_Corrected_Results.zip
The ZIP includes full source code in the code/ folder.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'__name__': '__main__',
 '__doc__': '\nComplete corrected WPT robustness simulation pipeline.\n\nWhat this corrected version adds compared with the basic roadmap code:\n1. C2 and L2 detuning sweeps.\n2. Frequency tracking under combined misalignment + detuning.\n3. Load optimization using three objectives:\n   - maximum efficiency,\n   - maximum output power,\n   - maximum efficiency subject to Pout >= target.\n4. Figures and CSVs that report efficiency and output power together.\n5. Robustness thresholds of 50%, 60%, and 70%, with explicit interpretation.\n6. Corrected coupling model naming: k_ref is coupling at the reference distance d_ref.\n7. The final ZIP includes full source code in the code/ folder.\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/WPT_Robustness_Project_Corrected/code/run_wpt_project.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\